# 01 · SFT v2: train in the same format you serve

**Decision this notebook supports:** if the model is trained on exactly the prompts it will see in use (system prompt + retrieved documents + question), and on near-miss refusals, do the v1 failures go away?

What changes from the main lab's notebook 05:

| | v1 (notebook 05) | v2 (here) |
|---|---|---|
| Prompt | short `SYSTEM`, no documents | the `ENGINEERED` prompt + retrieved chunks, byte-identical to evaluation |
| Data | 995 examples, refusals 94% spec/feature | 1,759 examples, refusals 35% money · 25% policy · rest spec/feature/company |
| Start point | base | base (a clean comparison, not stacked on v1) |
| Max length | 512 (would cut answers) | computed from the data so nothing is cut |
| LoRA | r 16 · α 32 · 7 modules | **same** (so only the data changed) |

Memory: gradient checkpointing is on, and the MPS allocator is capped (`configs/v2.yaml → memory_safety`). If the job runs out of memory you get an error, not a frozen Mac.

In [ ]:
# Find the v2 folder, put it (and the main lab one folder up) on the import path,
# and apply the memory-safety settings BEFORE torch is imported.
import sys, json, time, re, math, random, hashlib, collections
from pathlib import Path
V2_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "configs/v2.yaml").is_file())
sys.path.insert(0, str(V2_ROOT))
from v2lib import setup, device as D, prompts as P, facts as F
CFG = setup.cfg()
SNAP = setup.read_json(V2_ROOT / CFG["retrieval"]["snapshot"])
CHUNKS = SNAP["chunks"]
print("v2 folder:", V2_ROOT)
print("main lab:", setup.PARENT)
print("device:", D.device(), "| dtype:", D.dtype(), "| base model:", D.base_model_path())
D.memory_report("start")

## Load the base model

The original Qwen2.5-0.5B-Instruct in float32, the same precision as every v1 run.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
DEVICE, DTYPE = D.device(), D.dtype()
tok = AutoTokenizer.from_pretrained(D.base_model_path())
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(D.base_model_path(), dtype=DTYPE).to(DEVICE)
print(f"{sum(p.numel() for p in model.parameters())/1e6:.0f}M parameters on {DEVICE}")
D.memory_report("base loaded")

## Turn each example into tokens, and score only the answer

The chat template renders system + user + assistant. Everything up to and including `<|im_start|>assistant` gets the label `-100`, so it is ignored by the loss; only the answer (and its end marker) is learned.

The assert checks that the prompt tokens are an exact prefix of the full conversation, which is what makes this masking correct.

In [ ]:
train_rows = setup.read_jsonl(V2_ROOT / CFG["data"]["sft_train"])
val_rows = setup.read_jsonl(V2_ROOT / CFG["data"]["sft_val"])

def encode(e):
    msgs = P.messages(e["question"], [CHUNKS[c] for c in e["chunk_ids"]])
    prompt_ids = tok.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_dict=False)
    full_ids = tok.apply_chat_template(msgs + [{"role": "assistant", "content": e["answer"]}], tokenize=True, return_dict=False)
    assert full_ids[:len(prompt_ids)] == prompt_ids, "prompt must be an exact prefix"
    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
    return {"input_ids": full_ids, "attention_mask": [1] * len(full_ids), "labels": labels}

for name, e in [("RAG answer", next(x for x in train_rows if x["kind"] == "answer_with_gold_context")),
                ("near-miss refusal", next(x for x in train_rows if x["kind"] == "refuse_near_miss"))]:
    enc = encode(e)
    scored = [t for t in enc["labels"] if t != -100]
    print(f"{name}: {len(enc['input_ids'])} tokens, {len(scored)} scored")
    print("   scored text:", repr(tok.decode(scored)))

## Set `max_seq_len` from the data

Round the longest example up to a multiple of 64, then assert that no example is longer. An answer cut off at the end would teach the model half-sentences.

In [ ]:
train_enc = [encode(e) for e in train_rows]
val_enc = [encode(e) for e in val_rows]
longest = max(len(x["input_ids"]) for x in train_enc + val_enc)
MAX_LEN = int(math.ceil(longest / 64) * 64) if CFG["sft"]["max_seq_len"] == "auto" else int(CFG["sft"]["max_seq_len"])
assert all(len(x["input_ids"]) <= MAX_LEN for x in train_enc + val_enc), "an example would be truncated"
print(f"longest example {longest} tokens -> max_seq_len {MAX_LEN} (v1 used 512)")
print(f"train {len(train_enc)} | validation {len(val_enc)}")

In [ ]:
from torch.utils.data import Dataset

class Rows(Dataset):
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, i): return self.rows[i]

def collate(batch):
    n = max(len(b["input_ids"]) for b in batch)
    pad = tok.pad_token_id
    return {"input_ids": torch.tensor([b["input_ids"] + [pad] * (n - len(b["input_ids"])) for b in batch]),
            "attention_mask": torch.tensor([b["attention_mask"] + [0] * (n - len(b["attention_mask"])) for b in batch]),
            "labels": torch.tensor([b["labels"] + [-100] * (n - len(b["labels"])) for b in batch])}

train_ds, val_ds = Rows(train_enc), Rows(val_enc)
b = collate([train_enc[0], train_enc[1]])
print({k: tuple(v.shape) for k, v in b.items()}, "| padded labels ignored:", bool((b["labels"][b["attention_mask"] == 0] == -100).all()))

## Attach LoRA (same settings as v1)

Only the small adapter matrices train; the 494M base weights stay frozen.

In [ ]:
from peft import LoraConfig, get_peft_model
lc = CFG["lora"]
peft_cfg = LoraConfig(r=lc["r"], lora_alpha=lc["alpha"], lora_dropout=lc["dropout"],
                      target_modules=lc["target_modules"], task_type="CAUSAL_LM")
model = get_peft_model(model, peft_cfg)
model.print_trainable_parameters()

## Compute the loss only where it counts

A causal LM normally produces a score for **every** vocabulary word (151,936 for Qwen) at **every** position: about 650 positions × 152k × 4 bytes ≈ 400 MB per example, before gradients. The first attempt at this notebook hit the memory cap exactly there.

But only the answer positions are scored; every other label is `-100` and contributes nothing. So we ask the model for logits **only at the positions whose next token is scored** (`logits_to_keep`), then take the same cross-entropy. The loss is mathematically identical, as the next cell checks on a real example, and memory falls by about 15×.

In [ ]:
import torch.nn.functional as Fn
from transformers import Trainer

def answer_only_loss(m, batch, num_items_in_batch=None):
    labels = batch["labels"]
    shift = labels[:, 1:]                                   # token t+1 is predicted at position t
    keep = (shift != -100).any(dim=0).nonzero().squeeze(-1)  # positions whose next token is scored
    out = m(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"], logits_to_keep=keep)
    logits = out.logits.float()                            # [batch, len(keep), vocab]
    target = shift[:, keep]
    loss_sum = Fn.cross_entropy(logits.reshape(-1, logits.shape[-1]), target.reshape(-1), ignore_index=-100, reduction="sum")
    n = num_items_in_batch if num_items_in_batch is not None else (target != -100).sum()
    return loss_sum / n

class AnswerOnlyTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        loss = answer_only_loss(model, inputs, num_items_in_batch)
        return (loss, {"loss": loss}) if return_outputs else loss

# Equivalence check on one real example: full logits + HF loss vs answer-only logits.
probe = collate([train_enc[0]])
probe = {k: v.to(DEVICE) for k, v in probe.items()}
with torch.no_grad():
    full = model(**probe).loss
    mine = answer_only_loss(model, probe)
print(f"HF full-logits loss {full.item():.6f} | answer-only loss {mine.item():.6f} | difference {abs(full.item() - mine.item()):.2e}")
assert abs(full.item() - mine.item()) < 1e-4

## Training settings

Effective batch = batch 2 × accumulation 4 = 8 examples per weight update, as in v1. Gradient checkpointing recomputes activations during the backward pass instead of storing them, trading some speed for a much smaller memory peak.

In [ ]:
from transformers import Trainer, TrainingArguments
c = CFG["sft"]
args = TrainingArguments(
    output_dir=str(V2_ROOT / "runs/sft_v2_trainer"), num_train_epochs=c["epochs"], learning_rate=c["lr"],
    per_device_train_batch_size=c["batch_size"], per_device_eval_batch_size=c["batch_size"],
    gradient_accumulation_steps=c["grad_accum"], warmup_steps=c["warmup_ratio"],
    gradient_checkpointing=c["gradient_checkpointing"], gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=10, eval_strategy="epoch", save_strategy="no", report_to=[], seed=CFG["seed"],
    dataloader_num_workers=0, prediction_loss_only=True)
steps_per_epoch = math.ceil(len(train_ds) / (c["batch_size"] * c["grad_accum"]))
print(f"{steps_per_epoch} weight updates per epoch, {steps_per_epoch * c['epochs']} in total")
trainer = AnswerOnlyTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds, data_collator=collate)

## Train

This is the long cell. On this Mac, expect roughly 35–60 minutes.

In [ ]:
t0 = time.time()
result = trainer.train()
train_minutes = (time.time() - t0) / 60
print(result.metrics)
print(f"training took {train_minutes:.1f} minutes")
D.memory_report("after training")

## Loss curves

Training loss should fall smoothly. Validation loss is measured on held-out templates and **held-out refusal topics**, so a gap tells you how much is memorised phrasing rather than transferable behaviour.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
hist = pd.DataFrame(trainer.state.log_history)
fig, ax = plt.subplots(figsize=(8, 3.5))
hist.dropna(subset=["loss"]).plot(x="step", y="loss", ax=ax, label="train loss")
if "eval_loss" in hist:
    hist.dropna(subset=["eval_loss"]).plot(x="step", y="eval_loss", ax=ax, marker="o", label="validation loss")
ax.set_title("SFT v2"); plt.tight_layout(); plt.show()
display(hist.dropna(subset=["eval_loss"])[["epoch", "step", "eval_loss"]] if "eval_loss" in hist else hist.tail())

## Save the adapter and the run record

In [ ]:
out = V2_ROOT / CFG["sft"]["output_adapter"]
model.save_pretrained(str(out))
data_sha = {n: hashlib.sha256((V2_ROOT / CFG["data"][k]).read_bytes()).hexdigest() for n, k in [("train", "sft_train"), ("val", "sft_val")]}
setup.write_json(V2_ROOT / "runs/sft_v2.json", {
    "metrics": result.metrics, "train_minutes": round(train_minutes, 1), "max_seq_len": MAX_LEN,
    "log_history": trainer.state.log_history, "config": CFG["sft"], "lora": CFG["lora"],
    "n_train": len(train_ds), "n_val": len(val_ds), "data_sha256": data_sha, "device": DEVICE, "dtype": str(DTYPE),
    "saved_at": time.strftime("%Y-%m-%d %H:%M:%S")})
print("saved", out.relative_to(V2_ROOT), "and runs/sft_v2.json")

## Quick look: answers on held-out validation prompts

Greedy decoding, the same settings as the final evaluation. Four should be answered, four refused (two of them on refusal topics the model never trained on).

In [ ]:
model.eval()
def generate(m, question, chunk_ids, max_new_tokens=160):
    ids = tok.apply_chat_template(P.messages(question, [CHUNKS[c] for c in chunk_ids]), tokenize=True,
                                  add_generation_prompt=True, return_dict=False)
    x = torch.tensor([ids], device=DEVICE)
    with torch.inference_mode():
        out = m.generate(input_ids=x, attention_mask=torch.ones_like(x), max_new_tokens=max_new_tokens,
                         do_sample=False, num_beams=1, pad_token_id=tok.pad_token_id)
    return tok.decode(out[0, x.shape[1]:], skip_special_tokens=True).strip()

rng = random.Random(3)
picks = rng.sample([e for e in val_rows if e["kind"] == "answer_with_gold_context"], 4) + \
        rng.sample([e for e in val_rows if e["kind"] == "refuse_near_miss"], 2) + \
        rng.sample([e for e in val_rows if e["kind"] == "refuse_gold_removed"], 2)
for e in picks:
    print(f"[{e['kind']}] {e['question']}\n   model : {generate(model, e['question'], e['chunk_ids'])}\n   target: {e['answer']}\n")
D.memory_report("end")

<!-- sft-eval-section -->
## How good is SFT v2 on its own? Prompt only vs all documents vs RAG

Training loss says the model fits its data. This section asks the business question: **does it answer correctly and refuse appropriately?** Every dev_v1 (50) and dev_v2 (140) question is answered under four conditions:

| condition | what the model sees |
|---|---|
| `engineered` | the system prompt only: no documents (tests what it memorised) |
| `with_docs` | all five official documents pasted into the prompt (about 1,200 tokens; **not** the format it was trained on) |
| `rag_rerank` | the 3 retrieved chunks (the format it was trained on) |
| `oracle` | only the chunk that holds the answer (answerable questions only; the ceiling for RAG) |

Generation is identical to the main lab's notebook 13 and to v2 notebook 04 (greedy, 160 new tokens, float32). Answers are written into notebook 04's cache with the same keys, so the final evaluation reuses them. Scoring: the Gemini judge (primary) and the v1 keyword scorer (continuity).

In [ ]:
# Self-contained: loads SFT v2 from disk, so it does not depend on the training cells above.
import pandas as pd, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from src import lab
from v2lib import judge as J, gemini_io as G
EVAL = {"dev_v1": setup.read_json(setup.path(CFG["data"]["dev_v1"])), "dev_v2": setup.read_json(V2_ROOT / CFG["data"]["dev_v2"])}
DOCS = "\n\n".join(p.read_text() for p in sorted((setup.PARENT / "data/raw").glob("*.md")))   # the main lab's with_docs text
CONDS = ["engineered", "with_docs", "rag_rerank", "oracle"]
GEN = dict(CFG["generation"], device=D.device(), dtype=str(D.dtype()).replace("torch.", ""))
CACHE = V2_ROOT / CFG["final_eval"]["cache_dir"]; CACHE.mkdir(parents=True, exist_ok=True)

def chunk_ids_for(q, cond):
    if cond in ("engineered", "with_docs"):
        return []
    if cond == "rag_rerank":
        return SNAP["questions"][q["question"]]["rerank"][:3]
    return [c for c in sorted(CHUNKS) if q["answerable"] and q["fact_id"] in CHUNKS[c]["fact_ids"]]   # oracle

def system_for(cond, ids):
    return P.ENGINEERED + P.SEPARATOR + DOCS if cond == "with_docs" else P.system_prompt([CHUNKS[c] for c in ids])

adapter_dir = V2_ROOT / CFG["sft"]["output_adapter"]
IDENT = {"kind": "adapter", "adapter": hashlib.sha256((adapter_dir / "adapter_model.safetensors").read_bytes()).hexdigest()}
tok_e = AutoTokenizer.from_pretrained(D.base_model_path())
sft_model = PeftModel.from_pretrained(AutoModelForCausalLM.from_pretrained(D.base_model_path(), dtype=D.dtype()).to(D.device()),
                                      str(adapter_dir)).to(D.device()).eval()
jobs = [(cond, es, q) for cond in CONDS for es, items in EVAL.items() for q in items if cond != "oracle" or q["answerable"]]
print(len(jobs), "answers planned:", dict(collections.Counter(c for c, _, _ in jobs)))
D.memory_report("SFT v2 loaded for evaluation")

In [ ]:
def generate_answer(system, question):
    ids = tok_e.apply_chat_template([{"role": "system", "content": system}, {"role": "user", "content": question}],
                                    tokenize=True, add_generation_prompt=True, return_dict=False)
    assert len(ids) <= GEN["max_input_tokens"]
    x = torch.tensor([ids], device=D.device())
    eos = sft_model.generation_config.eos_token_id
    eos_ids = [e for e in (eos if isinstance(eos, list) else [eos]) if e is not None]
    with torch.inference_mode():
        out = sft_model.generate(input_ids=x, attention_mask=torch.ones_like(x), max_new_tokens=GEN["max_new_tokens"],
                                 do_sample=False, num_beams=1, eos_token_id=eos, pad_token_id=tok_e.pad_token_id or tok_e.eos_token_id)
    new = out[0, x.shape[1]:].tolist()
    return {"answer": tok_e.decode(new, skip_special_tokens=True).strip(), "completion_tokens": len(new), "prompt_tokens": len(ids),
            "truncated": len(new) >= GEN["max_new_tokens"] and (not new or new[-1] not in eos_ids)}

EV, t0 = [], time.time()
for n, (cond, es, q) in enumerate(jobs):
    ids = chunk_ids_for(q, cond)
    key_obj = {"model": IDENT, "cond": cond, "q": q["question"], "chunks": ids, "gen": GEN}          # notebook 04's cache key
    if cond == "with_docs":
        key_obj["system_sha256"] = P.sha(system_for(cond, ids))
    path = CACHE / (hashlib.sha256(json.dumps(key_obj, sort_keys=True).encode()).hexdigest() + ".json")
    if path.exists():
        rec = json.loads(path.read_text())
    else:
        rec = generate_answer(system_for(cond, ids), q["question"])
        rec.update(chunk_ids=ids, source="generated", model="sft_v2", condition=cond, eval_set=es, question_id=q["id"])
        path.write_text(json.dumps(rec, ensure_ascii=False))
    EV.append({"eval_set": es, "condition": cond, "question_id": q["id"], "answerable": q["answerable"], "category": q["category"],
               "style": q["style"], "question": q["question"], "answer": rec["answer"], "truncated": rec["truncated"],
               "prompt_tokens": rec.get("prompt_tokens")})
    if (n + 1) % 50 == 0:
        setup.progress("01_sft_eval", f"{n + 1}/{len(jobs)} answers | {(time.time() - t0) / 60:.1f} min")
print(f"{len(EV)} answers ready in {(time.time() - t0) / 60:.1f} min")
del sft_model; D.empty_cache(); D.memory_report("released")

### Score: Gemini judge (primary) and the v1 keyword scorer

The judge first passes its 60-item calibration check. Each distinct (question, answer) pair is judged once. "Accuracy" counts the right value even with an unsupported caveat; `caveat` shows how often such a caveat appeared.

In [ ]:
fidx = lab.fact_index()
QI = {(es, q["id"]): q for es, items in EVAL.items() for q in items}
cal = setup.read_json(V2_ROOT / CFG["data"]["judge_calibration"])
cal_res = J.judge(cal, purpose="judge_calibration", cap=150)
assert sum(cal_res[i["id"]]["label"] == i["expected"] for i in cal) / len(cal) >= CFG["judge"]["calibration_min_agreement"]
uniq = {}
for r in EV:
    q = QI[(r["eval_set"], r["question_id"])]
    s = lab.score_answer(r["answer"], q, fidx)
    r.update(legacy_correct=s["correct"], legacy_refused=s["refused"])
    r["jid"] = hashlib.sha256((r["eval_set"] + q["id"] + r["answer"]).encode()).hexdigest()[:20]
    uniq.setdefault(r["jid"], {"id": r["jid"], "question": q["question"], "answerable": q["answerable"],
                               "gold": J.gold_for(F.FIDX[q["fact_id"]]) if q["answerable"] else None,
                               "answer": r["answer"], "truncated": r["truncated"]})
print(f"{len(EV)} answers -> {len(uniq)} distinct pairs to judge")
setup.progress("01_sft_eval", f"judging {len(uniq)} distinct answers")
V = J.judge(list(uniq.values()), purpose="sft_v2_eval", cap=150)
E = pd.DataFrame(EV)
E["label"] = [V[j]["label"] for j in E.jid]
E["rationale"] = [V[j]["rationale"] for j in E.jid]
print(G.stats())

In [ ]:
def summarise(g):
    m = J.metrics(g[["answerable", "label"]].to_dict("records"))
    a, u = g[g.answerable], g[~g.answerable]
    return pd.Series({"n_ans": len(a), "n_unans": len(u), "accuracy": m["accuracy_answerable"], "caveat": m["caveat_rate_answerable"],
                      "hallucination": m["hallucination_rate_answerable"], "over_refusal": m["over_refusal_rate"],
                      "approp_refusal": m["appropriate_refusal_rate"], "invented_unans": m["invented_rate_unanswerable"],
                      "legacy_acc": a.legacy_correct.mean() if len(a) else None, "legacy_refusal": u.legacy_correct.mean() if len(u) else None,
                      "mean_prompt_tokens": g.prompt_tokens.mean()})
SFT_EVAL = E.groupby(["eval_set", "condition"]).apply(summarise, include_groups=False).reindex(CONDS, level="condition")
pd.set_option("display.width", 220)
display(SFT_EVAL.round(3))
setup.write_json(V2_ROOT / "runs/sft_v2_eval.json", {"scoreboard": json.loads(SFT_EVAL.reset_index().to_json(orient="records")),
                                                      "gemini": G.stats(), "saved_at": time.strftime("%Y-%m-%d %H:%M:%S")})

### Same questions, old ruler: SFT v1 vs SFT v2 on dev_v1

SFT v1's numbers come from the main lab's notebook 13 (keyword scorer). This cell is skipped if you have not run it. SFT v2 is shown on the same keyword scorer, so this row-for-row comparison uses one ruler.

In [ ]:
v1_csv = setup.path(CFG["final_eval"]["parent_v1_results"]) / "summary.csv"
if v1_csv.exists():
    v1 = pd.read_csv(v1_csv)
    v1 = v1[(v1.model == "sft") & v1.condition.isin(CONDS)].set_index("condition")[["accuracy_answerable", "appropriate_refusal_rate"]]
    v2 = SFT_EVAL.loc["dev_v1"][["legacy_acc", "legacy_refusal"]]
    cmp_ = pd.DataFrame({"SFT v1 accuracy": v1["accuracy_answerable"], "SFT v2 accuracy": v2["legacy_acc"],
                         "SFT v1 refusal": v1["appropriate_refusal_rate"], "SFT v2 refusal": v2["legacy_refusal"]}).reindex(CONDS)
    display(cmp_.round(3))
else:
    print("Skipped: run the main lab's notebook 13 (master evaluation) first to compare with SFT v1.")

### Refusals by category, and answers to read

Correct-refusal rate per unanswerable category (n in brackets), then a few wrong answers per condition from dev_v2.

In [ ]:
u = E[~E.answerable]
display(u.groupby(["eval_set", "category", "condition"]).apply(lambda g: f"{(g.label == 'refused').mean():.0%} ({len(g)})", include_groups=False)
        .unstack("condition").reindex(columns=[c for c in CONDS if c != "oracle"]))
def wrong(r):
    return r.label not in ("correct", "correct_with_unsupported_caveat") if r.answerable else r.label != "refused"
W = E[(E.eval_set == "dev_v2") & E.apply(wrong, axis=1)]
with pd.option_context("display.max_colwidth", None):
    display(W.groupby("condition").head(3)[["condition", "category", "question", "answer", "label"]])

### How to read this

- **`engineered` vs `rag_rerank`** shows what retrieval adds over memorised knowledge.
- **`with_docs` vs `rag_rerank`** shows whether three focused chunks beat five whole documents. SFT v2 trained only on the chunk format, so `with_docs` is partly out of distribution for it.
- **`oracle`** is the ceiling: with the right chunk alone, wrong answers are generation errors, not retrieval misses.
- **dev_v2 is the cleaner read.** dev_v1 informed the v2 design.
- **The full comparison** with base, v1 and DPO v2, plus the `rag_no_gold` condition, is in notebook 04.